# 🔤 Assamese OCR — Sentence Model Training

This notebook trains a CRNN (CNN + LSTM) model for Assamese sentence-level OCR using CTC loss.

**Before you start:**
1. Set runtime to **GPU** → Runtime > Change runtime type > T4 GPU
2. Upload your checkpoints to Google Drive (see Setup section)

---

## 1. Setup — Mount Drive & Clone Repo

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone your training repo (replace with your actual repo URL)
# If already cloned, skip this cell
REPO_URL = "https://github.com/rishav660/Assamese_OCR.git"  # ← CHANGE THIS
REPO_NAME = "Assamese_OCR"  # ← match your repo name
BRANCH = "develop"  # ← cloning the develop branch

import os
if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone -b {BRANCH} {REPO_URL}
else:
    print(f'Repo already exists at /content/{REPO_NAME}')
    # Pull latest changes from develop
    !cd /content/{REPO_NAME} && git checkout {BRANCH} && git pull

In [ ]:
# Set working directory
WORK_DIR = f'/content/{REPO_NAME}/django_backend'
%cd {WORK_DIR}
!pwd

In [ ]:
# Install dependencies
!pip install -q -r requirements-train.txt

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Link Drive Assets (Checkpoints & Data)

**First time only:** Create this folder structure in your Google Drive:
```
My Drive/
└── assamese_ocr_assets/
    ├── checkpoints/
    │   ├── best_model_sentences.pth
    │   └── best_model_fast.pth  (optional, for transfer learning)
    └── data/  (will be created by the split builder)
```

Upload your checkpoint files from `django_backend/checkpoints/` into that Drive folder.

In [ ]:
# Link Drive folders to expected local paths
DRIVE_ASSETS = '/content/drive/MyDrive/assamese_ocr_assets'

# Create Drive directories if they don't exist
os.makedirs(f'{DRIVE_ASSETS}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ASSETS}/data', exist_ok=True)

# Symlink checkpoints to Drive (so they persist)
!rm -rf checkpoints 2>/dev/null; ln -s {DRIVE_ASSETS}/checkpoints checkpoints

print('Symlinks created:')
!ls -la checkpoints
!ls -la data/

## 3. Generate Non-Overlapping Train/Val Splits

> **Important:** Data is generated to local disk first (fast), then copied to Drive for persistence.
> This avoids the Google Drive I/O errors that happen when writing thousands of small files via symlinks.

In [ ]:
# Check if splits already exist on Drive
train_labels = f'{DRIVE_ASSETS}/data/train_real_sentences/labels/labels.txt'
if os.path.exists(train_labels):
    with open(train_labels) as f:
        n = sum(1 for _ in f)
    print(f'✅ Splits already exist on Drive! ({n} training samples)')
    print('Skip the next cell unless you want to regenerate.')
else:
    print('❌ No splits found. Run the next cell to generate them.')

In [ ]:
# Step 1: Remove any Drive symlinks for data splits (generate locally)
import shutil

for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    dst = f'data/{split}'
    if os.path.islink(dst):
        os.unlink(dst)  # remove symlink
    elif os.path.exists(dst):
        shutil.rmtree(dst)  # remove old local dir
    os.makedirs(f'{dst}/images', exist_ok=True)
    os.makedirs(f'{dst}/labels', exist_ok=True)

print('✅ Local data directories ready')

# Step 2: Generate splits locally (fast, no Drive I/O issues)
!python build_real_sentence_splits.py \
    --input data/as-wiki-2021.txt \
    --train-output data/train_real_sentences \
    --val-output data/val_real_sentences \
    --test-output data/test_real_sentences \
    --train-count 50000 \
    --val-count 5000 \
    --test-count 2000 \
    --seed 42

In [ ]:
# Step 3: Copy generated data to Drive for persistence across sessions
import shutil

for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    src = f'data/{split}'
    dst = f'{DRIVE_ASSETS}/data/{split}'
    if os.path.exists(src) and os.listdir(f'{src}/images'):
        print(f'Copying {split} to Drive...')
        # Remove old Drive data if exists
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n_images = len(os.listdir(f'{dst}/images'))
        print(f'  ✅ {n_images} images copied')
    else:
        print(f'⚠️  {split}: no data to copy')

print('\n✅ All data persisted to Google Drive!')

## 4. Train the Sentence Model 🚀

In [ ]:
# Main training run
!python train_sentence_model.py \
    --train-img-dir data/train_real_sentences/images \
    --train-label-file data/train_real_sentences/labels/labels.txt \
    --val-img-dir data/val_real_sentences/images \
    --val-label-file data/val_real_sentences/labels/labels.txt \
    --epochs 25 \
    --batch-size 32 \
    --learning-rate 0.0001 \
    --best-checkpoint checkpoints/best_model_sentences.pth \
    --final-checkpoint checkpoints/final_model_sentences.pth \
    --plot-out training_curve_sentences.png

In [ ]:
# View training curve (now includes CER/WER plot)
from IPython.display import Image as IPImage, display
if os.path.exists('training_curve_sentences.png'):
    display(IPImage('training_curve_sentences.png'))
else:
    print('No training curve found yet.')

## 5. Evaluate Model Accuracy (CER / WER / Exact Match)

Run the evaluation script on your validation and test sets to measure accuracy properly.

In [ ]:
# Evaluate on VALIDATION set (raw model output, no spell check)
!python evaluate.py \
    --img-dir data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set (WITH spell-check post-processing)
# Compare results to see if spell checker helps or hurts
!python evaluate.py \
    --img-dir data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --post-process \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set using CTC Beam Search (Better Accuracy)
!python evaluate.py \
    --img-dir data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --beam-width 10 \
    --num-samples 10

In [ ]:
# Evaluate on TEST set (if generated)
test_labels = 'data/test_real_sentences/labels/labels.txt'
if os.path.exists(test_labels):
    !python evaluate.py \
        --img-dir data/test_real_sentences/images \
        --label-file {test_labels} \
        --checkpoint checkpoints/best_model_sentences.pth \
        --num-samples 10 \
        --output test_results.tsv
    print('\n✅ Detailed results saved to test_results.tsv')
else:
    print('❌ No test set found. Generate one with --test-count 1000')

In [ ]:
# Quick single-image prediction with ground-truth comparison
!python predict_cli.py \
    --image data/val_real_sentences/images/sentence_000000.png \
    --checkpoint checkpoints/best_model_sentences.pth \
    --ground-truth data/val_real_sentences/labels/sentence_000000.txt

## 6. (Optional) Transfer Learning

If you have `best_model_fast.pth` in `checkpoints/`, you can fine-tune from the character model.

In [ ]:
# Transfer learning (uses model_old.py architecture + best_model_fast.pth)
# !python train_transfer_learning.py

## 7. Save & Sync

Checkpoints auto-save to Drive via the symlink. To push code changes back:

In [ ]:
# Check checkpoint sizes on Drive
!ls -lh checkpoints/
print('\n✅ These are already on Drive — they persist across sessions.')

In [ ]:
# Push code changes back to GitHub (if you modified any .py files)
# %cd /content/{REPO_NAME}
# !git add -A
# !git commit -m "Colab training updates"
# !git push origin develop